<a href="https://colab.research.google.com/github/iiiyahiya111/pi_explore/blob/main/test_Dl_model_in_pi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **dl in raspberry pi**

In [1]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile

zip_path = Path(
    "/content/drive/MyDrive/"
    "Potato_Severity_Dataset_DEDUPED_FINAL.zip"
)

if not zip_path.exists():
    raise FileNotFoundError(
        f"Dataset ZIP not found: {zip_path}"
    )

with zipfile.ZipFile(zip_path, "r") as z:
    files = [
        name for name in z.namelist()
        if not name.endswith("/")
    ]

    print("Total files:", len(files))
    print("\nFirst 50 files:")
    for name in files[:50]:
        print(name)

    print("\nImage extensions:")
    from collections import Counter
    extensions = Counter(
        Path(name).suffix.lower()
        for name in files
    )
    print(extensions)


Mounted at /content/drive
Total files: 3330

First 50 files:
Potato_Severity_Dataset_FINAL/segmentation/valid/leaf_masks/40_Potato_Healthy_train_V2_JPG.rf.bd7863039286d033597c53d3cd6f05bd_leaf.png
Potato_Severity_Dataset_FINAL/segmentation/valid/leaf_masks/72_Potato_Healthy_train_V2_JPG.rf.6886d609356424a0d964f015515a2fd3_leaf.png
Potato_Severity_Dataset_FINAL/segmentation/valid/leaf_masks/95_Potato_Healthy_train_JPG.rf.962f8d34ce21064737ee304fa853ce5c_leaf.png
Potato_Severity_Dataset_FINAL/segmentation/valid/leaf_masks/77_Potato_Late_Blight_train_JPG.rf.077236aac90a79749779beeda0443593_leaf.png
Potato_Severity_Dataset_FINAL/segmentation/valid/leaf_masks/24_Potato_Healthy_val_V2_JPG.rf.fa93894baf5c70bb34480f1dfe6f2b6c_leaf.png
Potato_Severity_Dataset_FINAL/segmentation/valid/leaf_masks/40_Potato_Healthy_train_JPG.rf.4457409ba8dcd9d45b6af0771d2bb6c2_leaf.png
Potato_Severity_Dataset_FINAL/segmentation/valid/leaf_masks/29_Potato_Late_Blight_val_JPG.rf.c51848881aa782c9acfee37ee655f3ed_leaf

In [2]:

from collections import Counter, defaultdict
from pathlib import PurePosixPath
import zipfile

with zipfile.ZipFile(zip_path, "r") as z:
    names = [
        n for n in z.namelist()
        if n and not n.endswith("/")
    ]

    # 1. Count files by directory
    dir_counts = Counter()
    dir_examples = defaultdict(list)

    for name in names:
        p = PurePosixPath(name)
        parent = str(p.parent)
        ext = p.suffix.lower()

        if ext in {".jpg", ".jpeg", ".png", ".webp"}:
            dir_counts[parent] += 1
            if len(dir_examples[parent]) < 3:
                dir_examples[parent].append(p.name)

    print("=== IMAGE COUNTS BY DIRECTORY ===")
    for folder, count in sorted(
        dir_counts.items(),
        key=lambda x: (-x[1], x[0])
    ):
        print(f"\n{count:4d} images | {folder}")
        for sample in dir_examples[folder]:
            print("   ", sample)

    # 2. Show metadata files
    print("\n=== CSV / JSON FILES ===")
    for name in names:
        if PurePosixPath(name).suffix.lower() in {".csv", ".json"}:
            print(name)

    # 3. Summarize top-level split folders
    print("\n=== TOP-LEVEL STRUCTURE ===")
    roots = Counter()
    for name in names:
        parts = PurePosixPath(name).parts
        if len(parts) >= 2:
            roots["/".join(parts[:2])] += 1

    for folder, count in sorted(roots.items()):
        print(f"{count:4d} files | {folder}")


=== IMAGE COUNTS BY DIRECTORY ===

 895 images | Potato_Severity_Dataset_FINAL/segmentation/train/images
    282_Potato_Late_Blight_train_JPG.rf.18a26f164e210b00d10a4e610d91226d.jpg
    211_Potato_Late_Blight_train_JPG.rf.ff30f4ca3be6941f689655ce6a3ea729.jpg
    220_Potato_Healthy_train_JPG.rf.b856d159fe11cf93783a3cd23d5158b8.jpg

 895 images | Potato_Severity_Dataset_FINAL/segmentation/train/leaf_masks
    9_Potato_Late_Blight_val_V2_JPG.rf.28356e5f0d7da07945582bb6fa29907f_leaf.png
    96_Potato_Late_Blight_train_JPG.rf.27e73c469f50ecf627a3847979af30cc_leaf.png
    98_Potato_Late_Blight_train_JPG.rf.f0cc5d9e16842d3b47e69eb0f42bcc41_leaf.png

 895 images | Potato_Severity_Dataset_FINAL/segmentation/train/lesion_masks
    141_Potato_Healthy_train_JPG.rf.225a679c8388ea2cb1808ce3fc908f93.png
    183_Potato_Healthy_train_JPG.rf.630d277c0b8da084a357445631f1bae2.png
    314_Potato_Healthy_train_JPG.rf.3e715f78f348ed9fbf6e76cc1114d626.png

 107 images | Potato_Severity_Dataset_FINAL/segmentat

# **Leakage Test**

In [3]:

import re
import hashlib
import zipfile
import pandas as pd

from pathlib import PurePosixPath
from collections import Counter, defaultdict
from io import BytesIO

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

class_pattern = re.compile(
    r"_Potato_(Healthy|Late_Blight)_",
    re.IGNORECASE
)

records = []
unrecognized = []

with zipfile.ZipFile(zip_path, "r") as z:
    for info in z.infolist():
        if info.is_dir():
            continue

        p = PurePosixPath(info.filename)

        # Only original images, never leaf/lesion masks
        if (
            len(p.parts) < 5
            or p.parts[0] != "Potato_Severity_Dataset_FINAL"
            or p.parts[1] != "segmentation"
            or p.parts[3] != "images"
            or p.suffix.lower() not in IMAGE_EXTS
        ):
            continue

        split = p.parts[2]
        match = class_pattern.search(p.name)

        if not match:
            unrecognized.append(info.filename)
            continue

        label = (
            "Healthy"
            if match.group(1).lower() == "healthy"
            else "Late_Blight"
        )

        # Hash the image content, not its filename
        digest = hashlib.sha256()
        with z.open(info) as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                digest.update(chunk)

        records.append({
            "split": split,
            "label": label,
            "filename": p.name,
            "path": info.filename,
            "sha256": digest.hexdigest()
        })

df_images = pd.DataFrame(records)

print("=== IMAGE COUNTS BY SPLIT AND CLASS ===")
if not df_images.empty:
    print(
        df_images.groupby(["split", "label"])
        .size()
        .unstack(fill_value=0)
        .reindex(["train", "valid", "test"])
    )

print("\nTotal recognized images:", len(df_images))
print("Unrecognized filenames:", len(unrecognized))

if unrecognized:
    print("Examples:", unrecognized[:10])

# Check identical file content across splits
hash_groups = defaultdict(list)
for row in records:
    hash_groups[row["sha256"]].append(row)

duplicate_groups = [
    group for group in hash_groups.values()
    if len(group) > 1
]

cross_split_groups = [
    group for group in duplicate_groups
    if len({item["split"] for item in group}) > 1
]

print("\n=== EXACT DUPLICATE CHECK ===")
print("Duplicate hash groups, all splits:", len(duplicate_groups))
print("Duplicate hash groups across splits:", len(cross_split_groups))

for group in cross_split_groups[:10]:
    print("\nDuplicate group:")
    for item in group:
        print(f"  {item['split']}: {item['path']}")

# Inspect existing QC metadata for possible prior duplicate checks
with zipfile.ZipFile(zip_path, "r") as z:
    summary_path = (
        "Potato_Severity_Dataset_FINAL/"
        "qc/duplicate_removal_summary.json"
    )

    if summary_path in z.namelist():
        import json
        print("\n=== EXISTING DUPLICATE-REMOVAL SUMMARY ===")
        summary = json.loads(z.read(summary_path))
        print(json.dumps(summary, indent=2)[:5000])

    for csv_name in [
        "Potato_Severity_Dataset_FINAL/qc/duplicate_removal_manifest.csv",
        "Potato_Severity_Dataset_FINAL/qc/v5_FINAL_ALL_WITH_DECISION.csv"
    ]:
        if csv_name not in z.namelist():
            continue

        meta = pd.read_csv(BytesIO(z.read(csv_name)))
        print(f"\n=== {PurePosixPath(csv_name).name} ===")
        print("Shape:", meta.shape)
        print("Columns:", meta.columns.tolist())

        for col in meta.columns:
            if any(word in col.lower() for word in (
                "decision", "reason", "split",
                "status", "duplicate", "class", "label"
            )):
                print(
                    f"{col} value counts:",
                    meta[col].astype(str).value_counts().head(8).to_dict()
                )


=== IMAGE COUNTS BY SPLIT AND CLASS ===
label  Healthy  Late_Blight
split                      
train      445          450
valid       51           56
test        49           57

Total recognized images: 1108
Unrecognized filenames: 0

=== EXACT DUPLICATE CHECK ===
Duplicate hash groups, all splits: 0
Duplicate hash groups across splits: 0

=== EXISTING DUPLICATE-REMOVAL SUMMARY ===
{
  "criteria": "Strong pHash candidate followed by SSIM >= 0.985; high feature-match cases were used for confirmation.",
  "retention_policy": "One representative per confirmed duplicate component; prefer train, then valid, then test.",
  "duplicate_components": 34,
  "removed_images": 45,
  "kept_representatives": 34,
  "removed_filenames": [
    "13_Potato_Healthy_train_V2_JPG.rf.c82ae2344b4ddb1b17d017232b3edf1b.jpg",
    "156_Potato_Healthy_train_JPG.rf.38395e50e5e3c6885462a7af1062078a.jpg",
    "157_Potato_Healthy_train_JPG.rf.d48174329521253c70c44a6b0a012561.jpg",
    "17_Potato_Healthy_train_JPG.rf

# **QC যাচাই করে training dataset তৈরি**

In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path, PurePosixPath
from collections import Counter
from io import BytesIO
import zipfile
import shutil
import re
import pandas as pd

ZIP_PATH = Path(
    "/content/drive/MyDrive/"
    "Potato_Severity_Dataset_DEDUPED_FINAL.zip"
)

OUT_ROOT = Path("/content/potato_classification_data")

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"ZIP not found: {ZIP_PATH}")

QC_PATH = (
    "Potato_Severity_Dataset_FINAL/"
    "qc/v5_FINAL_ALL_WITH_DECISION.csv"
)
DUP_PATH = (
    "Potato_Severity_Dataset_FINAL/"
    "qc/duplicate_removal_manifest.csv"
)

pattern = re.compile(r"_Potato_(Healthy|Late_Blight)_",
                     re.IGNORECASE)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    all_names = set(z.namelist())

    for required in (QC_PATH, DUP_PATH):
        if required not in all_names:
            raise FileNotFoundError(
                f"Required QC file missing: {required}"
            )

    qc = pd.read_csv(BytesIO(z.read(QC_PATH)))
    dup = pd.read_csv(BytesIO(z.read(DUP_PATH)))

    qc["filename_key"] = qc["filename"].astype(str).map(
        lambda s: PurePosixPath(s).name
    )

    if qc["filename_key"].duplicated().any():
        repeated = qc.loc[
            qc["filename_key"].duplicated(keep=False),
            "filename_key"
        ].tolist()
        raise ValueError(
            f"Duplicate filenames in QC CSV: {repeated[:10]}"
        )

    qc_lookup = qc.set_index("filename_key").to_dict("index")

    remove_names = set(
        dup.loc[
            dup["decision"].astype(str).str.upper() == "REMOVE",
            "filename"
        ].astype(str).map(lambda s: PurePosixPath(s).name)
    )

    candidates = []

    for info in z.infolist():
        if info.is_dir():
            continue

        p = PurePosixPath(info.filename)

        if (
            len(p.parts) != 5
            or p.parts[0] != "Potato_Severity_Dataset_FINAL"
            or p.parts[1] != "segmentation"
            or p.parts[3] != "images"
            or p.suffix.lower() not in {".jpg", ".jpeg", ".png"}
        ):
            continue

        split = p.parts[2].lower()
        if split not in {"train", "valid", "test"}:
            continue

        match = pattern.search(p.name)
        if not match:
            continue

        label = (
            "Healthy" if match.group(1).lower() == "healthy"
            else "Late_Blight"
        )

        candidates.append({
            "zip_name": info.filename,
            "filename": p.name,
            "split": split,
            "label": label
        })

    selected = []
    missing_metadata = []
    metadata_mismatches = []
    skipped_reject = 0
    skipped_remove = 0

    for rec in candidates:
        name = rec["filename"]
        meta = qc_lookup.get(name)

        if meta is None:
            missing_metadata.append(name)
            continue

        decision = str(
            meta.get("final_decision", "")
        ).strip().upper()

        # Explicit QC rejection
        if decision != "KEEP":
            skipped_reject += 1
            continue

        # Exclude records designated REMOVE in duplicate manifest
        if name in remove_names:
            skipped_remove += 1
            continue

        expected_class = (
            "Potato Healthy"
            if rec["label"] == "Healthy"
            else "Potato Late Blight"
        )

        metadata_split = str(meta.get("split", "")).strip().lower()
        if metadata_split in {"val", "validation"}:
            metadata_split = "valid"

        metadata_class = str(
            meta.get("class_name", "")
        ).strip().casefold()

        if (
            metadata_split != rec["split"]
            or metadata_class != expected_class.casefold()
        ):
            metadata_mismatches.append({
                "filename": name,
                "folder_split": rec["split"],
                "csv_split": metadata_split,
                "expected_class": expected_class,
                "csv_class": metadata_class
            })
            continue

        selected.append(rec)

    print("Original image files:", len(candidates))
    print("Images skipped by QC:", skipped_reject)
    print("Images skipped by duplicate manifest:", skipped_remove)
    print("Images absent from QC metadata:", len(missing_metadata))
    print("Metadata mismatches:", len(metadata_mismatches))

    if missing_metadata:
        print("Missing metadata examples:", missing_metadata[:10])

    if metadata_mismatches:
        print("Mismatch examples:", metadata_mismatches[:10])

    if missing_metadata or metadata_mismatches:
        raise RuntimeError(
            "Dataset audit failed. Do not start training until "
            "the missing metadata or mismatches are investigated."
        )

    if not selected:
        raise RuntimeError("No approved training images remain.")

    # Rebuild folders from scratch to avoid stale files
    if OUT_ROOT.exists():
        shutil.rmtree(OUT_ROOT)

    for rec in selected:
        destination = (
            OUT_ROOT / rec["split"] / rec["label"] / rec["filename"]
        )
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(z.read(rec["zip_name"]))

counts = Counter((r["split"], r["label"]) for r in selected)

print("\n=== APPROVED IMAGE COUNTS ===")
for split in ("train", "valid", "test"):
    print(
        f"{split:5s} | "
        f"Healthy={counts[(split, 'Healthy')]:3d} | "
        f"Late_Blight={counts[(split, 'Late_Blight')]:3d}"
    )

print("\nPrepared dataset:", OUT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Original image files: 1108
Images skipped by QC: 0
Images skipped by duplicate manifest: 0
Images absent from QC metadata: 0
Metadata mismatches: 0

=== APPROVED IMAGE COUNTS ===
train | Healthy=445 | Late_Blight=450
valid | Healthy= 51 | Late_Blight= 56
test  | Healthy= 49 | Late_Blight= 57

Prepared dataset: /content/potato_classification_data


# **MobileNetV2 model train করা**

In [6]:
import tensorflow as tf
import numpy as np
from pathlib import Path

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

IMG_SIZE = 160
BATCH_SIZE = 16
CLASS_NAMES = ["Healthy", "Late_Blight"]

train_ds = tf.keras.utils.image_dataset_from_directory(
    str(OUT_ROOT / "train"),
    class_names=CLASS_NAMES,
    label_mode="int",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

valid_ds = tf.keras.utils.image_dataset_from_directory(
    str(OUT_ROOT / "valid"),
    class_names=CLASS_NAMES,
    label_mode="int",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    str(OUT_ROOT / "test"),
    class_names=CLASS_NAMES,
    label_mode="int",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
valid_ds = valid_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# Augmentation is active during training, not inference.
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
], name="data_augmentation")

# Pretrained lightweight feature extractor
base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    alpha=0.35,
    include_top=False,
    weights="imagenet",
    pooling="avg"
)

base.trainable = False

# Input pixels are 0..255.
# Preprocessing is embedded in the final model.
inputs = tf.keras.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3),
    dtype=tf.float32,
    name="image_rgb"
)

x = augmentation(inputs)
x = tf.keras.layers.Rescaling(
    scale=1.0 / 127.5,
    offset=-1.0,
    name="mobilenetv2_preprocessing"
)(x)

x = base(x, training=False)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(
    2, activation="softmax", name="classification"
)(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

print("\n=== Phase 1: Train classification head ===")
history_head = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=12,
    callbacks=callbacks
)

TensorFlow: 2.21.0
GPU: []
Found 895 files belonging to 2 classes.
Found 107 files belonging to 2 classes.
Found 106 files belonging to 2 classes.
2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_rgb (InputLayer)          │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_preprocessing       │ (None, 160, 160, 3)    │             0 │
│ (Rescaling)                     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_0.35_160            │ (None, 1280)           │       410,208 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification (Dense)          │ (None, 2)              │         2,562 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 412,770 (1.57 MB)

 Trainable params: 2,562 (10.01 KB)

 Non-trainable params: 410,208 (1.56 MB)


=== Phase 1: Train classification head ===
Epoch 1/12
56/56 ━━━━━━━━━━━━━━━━━━━━ 22s 296ms/step - accuracy: 0.8112 - loss: 0.4235 - val_accuracy: 0.9533 - val_loss: 0.1550 - learning_rate: 0.0010
Epoch 2/12
56/56 ━━━━━━━━━━━━━━━━━━━━ 18s 327ms/step - accuracy: 0.9475 - loss: 0.1623 - val_accuracy: 0.9626 - val_loss: 0.0907 - learning_rate: 0.0010
Epoch 3/12
56/56 ━━━━━━━━━━━━━━━━━━━━ 15s 259ms/step - accuracy: 0.9587 - loss: 0.1218 - val_accuracy: 0.9720 - val_loss: 0.0732 - learning_rate: 0.0010
Epoch 4/12
56/56 ━━━━━━━━━━━━━━━━━━━━ 15s 259ms/step - accuracy: 0.9642 - loss: 0.1095 - val_accuracy: 0.9813 - val_loss: 0.0643 - learning_rate: 0.0010
Epoch 5/12
56/56 ━━━━━━━━━━━━━━━━━━━━ 20s 260ms/step - accuracy: 0.9777 - loss: 0.0853 - val_accuracy: 0.9720 - val_loss: 0.0589 - learning_rate: 0.0010
Epoch 6/12
56/56 ━━━━━━━━━━━━━━━━━━━━ 21s 268ms/step - accuracy: 0.9832 - loss: 0.0716 - val_accuracy: 0.9907 - val_loss: 0.0503 - learning_rate: 0.0010
Epoch 7/12
56/56 ━━━━━━━━━━━━━━━━━━━━ 

In [7]:
# Unfreeze the later layers, but keep BatchNorm frozen.
base.trainable = True

for layer in base.layers:
    layer.trainable = False

for layer in base.layers[-20:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_finetune = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )
]

print("\n=== Phase 2: Fine-tuning ===")
history_finetune = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=8,
    callbacks=callbacks_finetune
)


=== Phase 2: Fine-tuning ===
Epoch 1/8
56/56 ━━━━━━━━━━━━━━━━━━━━ 24s 311ms/step - accuracy: 0.9687 - loss: 0.0716 - val_accuracy: 0.9907 - val_loss: 0.0436
Epoch 2/8
56/56 ━━━━━━━━━━━━━━━━━━━━ 16s 286ms/step - accuracy: 0.9754 - loss: 0.0667 - val_accuracy: 0.9813 - val_loss: 0.0424
Epoch 3/8
56/56 ━━━━━━━━━━━━━━━━━━━━ 16s 281ms/step - accuracy: 0.9754 - loss: 0.0633 - val_accuracy: 0.9907 - val_loss: 0.0387
Epoch 4/8
56/56 ━━━━━━━━━━━━━━━━━━━━ 20s 269ms/step - accuracy: 0.9844 - loss: 0.0461 - val_accuracy: 0.9907 - val_loss: 0.0421
Epoch 5/8
56/56 ━━━━━━━━━━━━━━━━━━━━ 20s 265ms/step - accuracy: 0.9788 - loss: 0.0499 - val_accuracy: 0.9907 - val_loss: 0.0322
Epoch 6/8
56/56 ━━━━━━━━━━━━━━━━━━━━ 20s 268ms/step - accuracy: 0.9799 - loss: 0.0525 - val_accuracy: 0.9907 - val_loss: 0.0393
Epoch 7/8
56/56 ━━━━━━━━━━━━━━━━━━━━ 15s 267ms/step - accuracy: 0.9855 - loss: 0.0368 - val_accuracy: 0.9907 - val_loss: 0.0278
Epoch 8/8
56/56 ━━━━━━━━━━━━━━━━━━━━ 15s 274ms/step - accuracy: 0.9832 - l

# **Test evaluation ও .tflite export**

In [8]:
import json
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Test set has shuffle=False, so labels and predictions stay aligned.
y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
])

probabilities = model.predict(test_ds, verbose=1)
y_pred = np.argmax(probabilities, axis=1)

DISPLAY_NAMES = ["Healthy", "Late Blight"]

print("\n=== FINAL TEST RESULTS ===")
print("Test images:", len(y_true))
print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=DISPLAY_NAMES,
    digits=4,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
print("Confusion matrix (rows=true, columns=predicted):")
print(cm)

# Export folder in Google Drive
OUT_DIR = Path(
    "/content/drive/MyDrive/Potato_Zero2W_Deployment"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the trainable Keras model for future improvements.
model.save(str(OUT_DIR / "potato_mobilenetv2.keras"))

# Convert to TFLite using float32 input/output for the first Pi test.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()

MODEL_PATH = OUT_DIR / "potato_mobilenetv2.tflite"
MODEL_PATH.write_bytes(tflite_bytes)

(OUT_DIR / "labels.txt").write_text(
    "Healthy\nLate Blight\n",
    encoding="utf-8"
)

metadata = {
    "architecture": "MobileNetV2",
    "alpha": 0.35,
    "input_shape": [1, IMG_SIZE, IMG_SIZE, 3],
    "input_dtype": "float32",
    "input_pixel_range": [0, 255],
    "preprocessing": (
        "Embedded in model: scale pixels by 1/127.5 "
        "and offset by -1.0"
    ),
    "class_order": DISPLAY_NAMES,
    "model_file": MODEL_PATH.name
}

(OUT_DIR / "model_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

# Save test predictions and metrics.
report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=DISPLAY_NAMES,
    output_dict=True,
    zero_division=0
)

metrics = {
    "test_images": int(len(y_true)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "confusion_matrix": cm.tolist(),
    "classification_report": report
}

(OUT_DIR / "test_metrics.json").write_text(
    json.dumps(metrics, indent=2),
    encoding="utf-8"
)

pd.DataFrame({
    "true_label": [DISPLAY_NAMES[i] for i in y_true],
    "predicted_label": [DISPLAY_NAMES[i] for i in y_pred],
    "prob_healthy": probabilities[:, 0],
    "prob_late_blight": probabilities[:, 1],
    "correct": (y_true == y_pred)
}).to_csv(OUT_DIR / "test_predictions.csv", index=False)

# Verify TFLite inference inside Colab.
interpreter = tf.lite.Interpreter(
    model_content=tflite_bytes,
    num_threads=2
)
interpreter.allocate_tensors()

input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]

sample_images, _ = next(iter(test_ds))
sample = sample_images[:1].numpy().astype(
    input_detail["dtype"]
)

interpreter.set_tensor(input_detail["index"], sample)
interpreter.invoke()

tflite_probs = interpreter.get_tensor(
    output_detail["index"]
)
keras_probs = model(sample, training=False).numpy()

print("\n=== TFLITE SMOKE TEST ===")
print("Input shape:", input_detail["shape"])
print("Input dtype:", input_detail["dtype"])
print("Keras probabilities:", keras_probs[0])
print("TFLite probabilities:", tflite_probs[0])
print(
    "Maximum output difference:",
    float(np.max(np.abs(keras_probs - tflite_probs)))
)

print("\nModel file:", MODEL_PATH)
print("Model size (MB):", round(MODEL_PATH.stat().st_size / 1e6, 2))
print("All deployment files:", OUT_DIR)

7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step

=== FINAL TEST RESULTS ===
Test images: 106
Accuracy: 0.9717

Classification report:
              precision    recall  f1-score   support

     Healthy     0.9600    0.9796    0.9697        49
 Late Blight     0.9821    0.9649    0.9735        57

    accuracy                         0.9717       106
   macro avg     0.9711    0.9723    0.9716       106
weighted avg     0.9719    0.9717    0.9717       106

Confusion matrix (rows=true, columns=predicted):
[[48  1]
 [ 2 55]]
Saved artifact at '/tmp/tmpxfulc8bi'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name='image_rgb')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  139731968167760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139731968167952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139731968170064: TensorSpec(shape=(), dtype=tf.resource, na

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



=== TFLITE SMOKE TEST ===
Input shape: [  1 160 160   3]
Input dtype: <class 'numpy.float32'>
Keras probabilities: [9.9963975e-01 3.6023164e-04]
TFLite probabilities: [9.9963975e-01 3.6022757e-04]
Maximum output difference: 4.0745362639427185e-09

Model file: /content/drive/MyDrive/Potato_Zero2W_Deployment/potato_mobilenetv2.tflite
Model size (MB): 1.63
All deployment files: /content/drive/MyDrive/Potato_Zero2W_Deployment


# **Raspberry Pi-তে নেওয়ার জন্য package তৈরি**

In [9]:
package_path = OUT_DIR / "potato_zero2w_package.zip"

with zipfile.ZipFile(
    package_path, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for filename in [
        "potato_mobilenetv2.tflite",
        "labels.txt",
        "model_metadata.json"
    ]:
        file_path = OUT_DIR / filename
        if not file_path.exists():
            raise FileNotFoundError(file_path)
        z.write(file_path, arcname=filename)

print("Deployment package:", package_path)
print("Package size (MB):", round(package_path.stat().st_size / 1e6, 2))

Deployment package: /content/drive/MyDrive/Potato_Zero2W_Deployment/potato_zero2w_package.zip
Package size (MB): 1.47


# **Colab থেকে test dataset ZIP তৈরি করো**

In [10]:
from pathlib import Path
import zipfile

TEST_ROOT = Path("/content/potato_classification_data/test")
ZIP_PATH = Path("/content/potato_test_images.zip")

if not TEST_ROOT.is_dir():
    raise FileNotFoundError(
        f"Test folder পাওয়া যায়নি: {TEST_ROOT}\n"
        "Dataset তৈরির cell-এ ব্যবহৃত প্রকৃত path পরীক্ষা করো।"
    )

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
images = [
    p for p in TEST_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in image_exts
]

print("Test image count:", len(images))

for folder in sorted(p for p in TEST_ROOT.iterdir() if p.is_dir()):
    count = sum(1 for p in folder.rglob("*")
                if p.is_file() and p.suffix.lower() in image_exts)
    print(f"{folder.name}: {count}")

if not images:
    raise RuntimeError("Test folder-এ কোনো image পাওয়া যায়নি।")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for image_path in images:
        # ZIP-এর মধ্যে Healthy/filename.jpg-এর মতো structure থাকবে
        z.write(image_path, arcname=image_path.relative_to(TEST_ROOT))

print("Created:", ZIP_PATH)
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / (1024 * 1024), 2))

from google.colab import files
files.download(str(ZIP_PATH))

Test image count: 106
Healthy: 49
Late_Blight: 57
Created: /content/potato_test_images.zip
ZIP size (MB): 1.82


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>